# Explore bed and doctor counts before modeling

Read `linked_fdr` from notebook 02. Audit bed capacity and number of doctors
independently. Review the observed values, the reporting gaps, and plausible
inputs before choosing a regression specification. This notebook performs
descriptive analysis and saves tables for review.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## What the literature changes here

Examine zero values, dispersion and differences across facility types before
choosing a count model. The shape of the outcome can affect count-model accuracy.
A marginal variance-to-mean ratio alone cannot assess a conditional model.
[Alomair (2024)](https://pmc.ncbi.nlm.nih.gov/articles/PMC11687910/)

Spatial structure can make random validation optimistic. This audit keeps linked
facility IDs together. Hold out whole geographies later if the goal is transfer
to unseen districts and the data support that split.
[Linnenbrink et al. (2024)](https://gmd.copernicus.org/articles/17/5897/2024/gmd-17-5897-2024.html)

Assess missing-field reconstruction and downstream recommendations separately.
An imputation benchmark found that lower reconstruction error did not necessarily
improve downstream performance. Compare reporting rates before treating labeled
facilities as representative, then measure what estimates change in the shortlist.
[Nikitina and Mouriño (2026)](https://link.springer.com/article/10.1007/s41060-026-01295-3)

For future service classification, use independently reviewed positives and negatives.
Text silent on a service stays unlabeled.

In [ ]:
from facility_tool.eda import COUNT_UNITS, capacity_eda

linked = STORE.read("linked_fdr")
entities = STORE.read("entities")
links = STORE.read("links")
records = STORE.read("records")
original = [field for field in linked.columns if field != "facility_id"]
seed = 42
# Example override per target: ["facilityTypeId", "operatorTypeId"].
feature_columns = {"capacity": None, "numberDoctors": None}

# Extra context is for reporting-gap audits. Keep it out of the original schema list.
sources = records[["record_id", "source"]].merge(links[["record_id", "facility_id"]], on="record_id", validate="1:1")
sources = sources.loc[sources["facility_id"].fillna("").ne("")].groupby("facility_id")["source"].agg(
    lambda names: ", ".join(sorted(set(names)))
)
audits = linked.copy()
audits["source_types"] = audits["facility_id"].map(sources)
branches = entities.set_index("facility_id")
for field in ("state", "district"):
    if field not in original:
        places = branches[field].replace("", None).fillna(branches[f"postal_{field}"])
        places = places.mask(branches["postal_resolution"].isin(["state_conflict", "district_conflict"]))
        audits[field] = audits["facility_id"].map(places)

## Label quality and the proposed holdout

The parser separates exact counts, ranges, unclear units, implausible numeric
forms and missing values. Review conflicting values from linked rows as a branch
issue. Facility IDs define the proposed train, calibration and test partitions.
Whole-snapshot reports below describe coverage. Relationships and plots use
training rows only. The readiness counts help identify small cohorts before fitting.

In [ ]:
results = {}
names = {"capacity": "capacity", "numberDoctors": "number_doctors"}
for target, prefix in names.items():
    if target not in original:
        print(f"{target}: no original facility field loaded in {MODE} mode.")
        continue
    result = capacity_eda(audits, target, original, seed=seed, feature_columns=feature_columns[target],
                          reference_year=int(config["as_of"][:4]))
    results[target] = result
    for table, frame in result.items():
        STORE.write(f"{prefix}_{table}", frame)
    print(f"{target} ({COUNT_UNITS[target]})")
    show(result["summary"])
    print("Source-level parsing outcomes")
    show(result["source_target_audit"].groupby("parse_status").size().rename("source_rows").reset_index())
    print("Branch-level conflicts and unclear counts")
    review = result["target_audit"]
    show(review.loc[review["status"].isin(["conflicting", "unparseable"])])
    print("Proposed holdout readiness")
    show(result["split_readiness"])

## Reporting gaps, leakage and interactions

Inspect counts alongside percentages and correlations. A pattern in a group with
few observed counts can be unstable. A potential predictor must appear after
`countries` in the source schema. The structured subset limits obvious leakage.
Text that repeats the target and suspicious count agreement deserve a source check.

In [ ]:
for target, result in results.items():
    for table in ("missingness", "feature_audit", "group_coverage", "numeric_relationships",
                  "categorical_relationships", "interaction_summary"):
        print(f"{target} | {table.replace('_', ' ').title()}")
        frame = result[table]
        if table == "missingness":
            frame = frame.sort_values("raw_missing_fraction", ascending=False)
        show(frame, limit=60 if table in ("missingness", "feature_audit") else 20)

## Training label distributions

These charts use exact training counts. The histogram and quantiles show tail
behavior and sparse or zero counts. The scatter plot uses only paired observations.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

for target, result in results.items():
    training = result["analysis_rows"]
    observed = pd.to_numeric(training["value"], errors="coerce").dropna()
    if observed.empty:
        continue
    print(f"{target} | Training summary")
    show(observed.describe(percentiles=[.1, .25, .5, .75, .9, .99]).rename(COUNT_UNITS[target]))
    print(f"Explicit zeros: {observed.eq(0).sum():,}")
    if len(observed) > 1 and observed.mean() > 0:
        print(f"Marginal variance / mean: {observed.var() / observed.mean():.2f}")
    if plt is None:
        continue
    other = next(field for field in COUNT_UNITS if field != target)
    plot_pair = other in training and training[["value", other]].notna().all(axis=1).any()
    figure, axes = plt.subplots(1, 1 + int(plot_pair), figsize=(10 if plot_pair else 5, 3.5), squeeze=False)
    axes[0, 0].hist(observed, bins="auto", color="#24576d")
    axes[0, 0].set(xlabel=f"Reported {COUNT_UNITS[target]}", ylabel="Training facilities")
    if plot_pair:
        paired = training[["value", other]].dropna()
        axes[0, 1].scatter(paired[other], paired["value"], alpha=.5, color="#24576d")
        axes[0, 1].set(xlabel=f"Reported {COUNT_UNITS[other]}", ylabel=f"Reported {COUNT_UNITS[target]}")
    figure.tight_layout()
    plt.show()
    plt.close(figure)